# 02 — LLM basics: tokens, siguiente token y generación

## Segundo notebook del bloque LLM

En el notebook anterior utilizamos un modelo de embeddings para representar textos y recuperar información semánticamente relacionada.

Ahora veremos otro tipo de modelo: un **causal language model**, cuyo mecanismo básico consiste en predecir el siguiente token.

```text
texto → tokenizer → token IDs → LLM → logits → probabilidades → siguiente token → repetir
```

No entrenaremos el modelo. Usaremos uno pequeño ya preentrenado para inspeccionar el proceso localmente.

## 0. Dependencias

Si fuera necesario, desde la raíz del proyecto:

```bash
uv add transformers torch
```

Usaremos `distilbert/distilgpt2`. Es un modelo pequeño, principalmente orientado al inglés. Lo elegimos por su tamaño y valor didáctico, no porque sea el modelo que usaríamos en una aplicación ATM real en español.

## 1. Importaciones

In [ ]:
import torch
import pandas as pd

from transformers import AutoTokenizer, AutoModelForCausalLM

pd.set_option("display.max_colwidth", 120)

## 2. Cargamos tokenizer y modelo

In [ ]:
MODEL_NAME = "distilbert/distilgpt2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.eval()

print("Modelo:", MODEL_NAME)
print("Parámetros:", f"{model.num_parameters():,}")

`AutoTokenizer` convierte texto en IDs de tokens. `AutoModelForCausalLM` carga un modelo preparado para **causal language modeling**: producir una distribución para el siguiente token a partir de los anteriores.

## 3. Tokenización: el modelo no recibe palabras

In [ ]:
text = "Air traffic is increasing rapidly."
encoded = tokenizer(text, return_tensors="pt")
input_ids = encoded["input_ids"]

print("Shape:", input_ids.shape)
print("Token IDs:", input_ids[0].tolist())

pd.DataFrame({
    "token_id": input_ids[0].tolist(),
    "token": tokenizer.convert_ids_to_tokens(input_ids[0]),
})

Un token no equivale necesariamente a una palabra. Puede ser una palabra, parte de una palabra, puntuación o un fragmento frecuente.

## 4. ¿Qué ocurre con español?

In [ ]:
examples = [
    "El tráfico aéreo está aumentando rápidamente.",
    "La capacidad del sector se ha reducido.",
    "cumulonimbos",
]

rows = []
for example in examples:
    ids = tokenizer(example)["input_ids"]
    rows.append({
        "text": example,
        "n_characters": len(example),
        "n_tokens": len(ids),
        "tokens": tokenizer.convert_ids_to_tokens(ids),
    })

pd.DataFrame(rows)

Un tokenizer poco adaptado a un idioma puede fragmentar más ciertas palabras. Esto afecta al tamaño efectivo del contexto y al coste de procesarlo.

# 5. Forward pass: del prompt a los logits

In [ ]:
prompt = "Air traffic control must"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print("Logits shape:", outputs.logits.shape)

La forma es aproximadamente:

```text
(batch_size, sequence_length, vocabulary_size)
```

Para cada posición, el modelo produce un valor para cada token posible del vocabulario. Esos valores se llaman **logits**.

## 6. De logits a probabilidades del siguiente token

In [ ]:
next_token_logits = outputs.logits[0, -1]
next_token_probs = torch.softmax(next_token_logits, dim=-1)

print("Número de tokens candidatos:", next_token_probs.shape[0])
print("Suma de probabilidades:", next_token_probs.sum().item())

Conceptualmente estamos calculando:

\[
P(token_{t+1} \mid token_1, token_2, ..., token_t)
\]

## 7. Los 10 siguientes tokens más probables

In [ ]:
top_probs, top_ids = torch.topk(next_token_probs, 10)

rows = []
for probability, token_id in zip(top_probs, top_ids):
    rows.append({
        "token_id": token_id.item(),
        "token": tokenizer.decode([token_id.item()]),
        "probability": probability.item(),
    })

pd.DataFrame(rows)

El modelo no contiene una única continuación prefijada: produce una distribución de probabilidad sobre muchas continuaciones posibles.

# 8. Generación autoregresiva manual

In [ ]:
manual_ids = tokenizer("Air traffic control", return_tensors="pt")["input_ids"]
steps = []

for step in range(8):
    with torch.no_grad():
        outputs = model(input_ids=manual_ids)

    logits = outputs.logits[0, -1]
    probs = torch.softmax(logits, dim=-1)
    new_token_id = torch.argmax(probs).view(1, 1)

    steps.append({
        "step": step + 1,
        "selected_token": repr(tokenizer.decode(new_token_id[0])),
        "probability": probs[new_token_id.item()].item(),
    })

    manual_ids = torch.cat([manual_ids, new_token_id], dim=1)

print(tokenizer.decode(manual_ids[0]))
pd.DataFrame(steps)

Acabamos de implementar una versión mínima de generación autoregresiva:

```text
tokens existentes → modelo → siguiente token → añadirlo → repetir
```

# 9. `generate()` hace este trabajo por nosotros

In [ ]:
prompt = "Air traffic management requires"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))

Con `do_sample=False` utilizamos una generación determinista de tipo greedy: se favorece la continuación de máxima puntuación en cada paso.

# 10. Sampling: no elegir siempre el token número 1

In [ ]:
prompt = "Air traffic management requires"
inputs = tokenizer(prompt, return_tensors="pt")

for i in range(3):
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    print(f"--- Generation {i + 1} ---")
    print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))
    print()

`temperature`, `top_k` y `top_p` modifican **cómo elegimos tokens de la distribución**. No reentrenan el modelo ni añaden conocimiento.

## 11. Temperature

In [ ]:
prompt = "The safest way to manage heavy air traffic is"
inputs = tokenizer(prompt, return_tensors="pt")

for temperature in [0.3, 0.8, 1.3]:
    torch.manual_seed(42)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=25,
            do_sample=True,
            temperature=temperature,
            top_k=50,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    print(f"TEMPERATURE = {temperature}")
    print(tokenizer.decode(generated_ids[0], skip_special_tokens=True))
    print()

Intuición:

```text
temperatura baja → distribución más concentrada → salida más conservadora
temperatura alta → distribución más plana → mayor variedad y riesgo
```

# 12. El contexto cambia la predicción

In [ ]:
def top_next_tokens(prompt, k=5):
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        logits = model(**inputs).logits[0, -1]

    probs = torch.softmax(logits, dim=-1)
    top_probs, top_ids = torch.topk(probs, k)

    return pd.DataFrame({
        "token": [tokenizer.decode([token_id.item()]) for token_id in top_ids],
        "probability": [probability.item() for probability in top_probs],
    })

prompts = [
    "The aircraft is approaching the",
    "The storm is approaching the",
    "The controller is approaching the",
]

for prompt in prompts:
    print("=" * 80)
    print(prompt)
    display(top_next_tokens(prompt).round(4))

El contexto anterior modifica la distribución del siguiente token. Esa dependencia contextual es una pieza esencial de los Transformers.

# 13. ¿Dónde están los embeddings ahora?

In [ ]:
embedding_layer = model.get_input_embeddings()

print("Vocabulary size:", embedding_layer.weight.shape[0])
print("Embedding dimension:", embedding_layer.weight.shape[1])

Diferencia conceptual:

### Notebook 01: embeddings para búsqueda

```text
documento → vector → comparar con otros textos
```

### Dentro de un LLM

```text
token embeddings → Transformer → representaciones contextualizadas → siguiente token
```

# 14. Modelo de lenguaje ≠ base de datos

In [ ]:
prompt = "The fictional Madrid North Oceanic Control Centre was founded in"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=30,
        do_sample=True,
        temperature=0.8,
        top_p=0.95,
        pad_token_id=tokenizer.eos_token_id,
    )

print(tokenizer.decode(output[0], skip_special_tokens=True))

La entidad del prompt es deliberadamente inventada. El modelo puede intentar continuarla igualmente porque optimiza plausibilidad lingüística, no una consulta verificable a una base de datos.

Ésta es una de las motivaciones de RAG:

```text
LLM solo → puede producir algo plausible pero incorrecto
RAG → recuperar fuentes → añadir contexto → generar respuesta apoyada en información concreta
```

# 15. Prompt, modelo y decoding son piezas distintas

Conviene separar:

1. **Prompt**: tokens que proporcionamos como contexto.
2. **Modelo**: red y parámetros aprendidos durante pretraining.
3. **Decoding**: estrategia para elegir tokens durante la generación (`greedy`, sampling, temperature, top-k, top-p...).

Cambiar el decoding **no modifica los parámetros del modelo**.

# 16. Conexión con lo aprendido

```text
texto
  ↓
tokens
  ↓
token embeddings
  ↓
Transformer / self-attention
  ↓
logits
  ↓
softmax
  ↓
P(siguiente token)
  ↓
decoding
  ↓
texto generado
```

Y desde el notebook anterior:

```text
documentos → embeddings → semantic search → contexto
```

Pronto uniremos ambos caminos:

```text
semantic search → contexto + pregunta → LLM → respuesta
```

# 17. Pequeños experimentos

1. Cambia el prompt y examina los 10 siguientes tokens.
2. Prueba una frase ambigua y modifica solo unas palabras del contexto.
3. Compara `do_sample=False` con `do_sample=True`.
4. Ejecuta generaciones con varias temperaturas.
5. Prueba una entidad ficticia y observa si el modelo intenta continuarla.

No buscamos calidad de respuesta: DistilGPT2 es deliberadamente pequeño y principalmente inglés. Buscamos entender el mecanismo.

# 18. Preguntas de comprobación

Al terminar deberías poder responder:

1. ¿Qué diferencia existe entre palabra y token?
2. ¿Qué son los `input_ids`?
3. ¿Qué son los logits?
4. ¿Por qué aplicamos softmax?
5. ¿Qué significa `P(siguiente token | tokens anteriores)`?
6. ¿Qué significa generación autoregresiva?
7. ¿Qué diferencia existe entre greedy y sampling?
8. ¿Qué modifica la temperature?
9. ¿Cambiar temperature significa reentrenar?
10. ¿Por qué el mismo modelo puede generar textos distintos?
11. ¿Por qué un texto plausible puede ser falso?
12. ¿Qué diferencia hay entre los embeddings del notebook 01 y las representaciones internas de tokens de un LLM?
13. ¿Qué papel juega el contexto en la predicción del siguiente token?

# 19. Siguiente paso

Ya tenemos dos piezas:

```text
01_embeddings_semantic_search → recuperar información
02_llm_basics                 → generar texto
```

El siguiente notebook utilizará un **LLM moderno como componente de una aplicación**: instrucciones, prompt, contexto y respuesta. Después conectaremos esa llamada con `retrieve()` para construir nuestro primer **RAG ATM**.